# CNN Development on a Custom Dataset
**Course:** CVPR — Mid Assignment (Summer 25-26)
**Student ID:** `23-50953-1`

**Dataset used:** [Beans Leaf Disease Dataset](https://huggingface.co/datasets/beans) (Hugging Face) — 3 classes: `healthy`, `angular_leaf_spot`, `bean_rust`. The dataset ships with pre-defined **train / validation / test** splits, ~1,295 images total, which makes it a good size for a from-scratch CNN on a single GPU/CPU session.

**Why this dataset:** it is small enough to train quickly, has a clean multi-class (3-class) structure, real-world (non-toy) images, and is directly loadable with the `datasets` library (no manual download / API key needed), which keeps the notebook fully reproducible.

> **How to run:** Open this notebook in **Google Colab** (Runtime → Change runtime type → GPU) and run all cells top to bottom. First run will download the dataset automatically (~1,300 small JPEGs, <100MB).


## 1. Import Libraries

In [ ]:
# Install packages not pre-installed on Colab (safe to re-run)
!pip install -q datasets torchinfo


In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)

from datasets import load_dataset
from torchinfo import summary

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## 2. Load and Explore Dataset

The `beans` dataset comes with three splits already defined: `train`, `validation`, and `test`. We load all three directly from the Hugging Face Hub.

In [ ]:
# NOTE: the original "beans" repo used a Python loading script, which the
# current `datasets` library (>=4.0) refuses to execute at all -> RuntimeError:
# "Dataset scripts are no longer supported". The line below instead uses the
# maintainer's re-upload with the script removed and plain Parquet files
# committed directly, so it loads normally on any recent `datasets` version.
raw_datasets = load_dataset("AI-Lab-Makerere/beans")
print(raw_datasets)

class_names = raw_datasets["train"].features["labels"].names
num_classes = len(class_names)
print("Classes:", class_names)
print("Number of classes:", num_classes)
print("Train size:", len(raw_datasets["train"]))
print("Validation size:", len(raw_datasets["validation"]))
print("Test size:", len(raw_datasets["test"]))


In [ ]:
# Explore class balance in the training split
train_labels = raw_datasets["train"]["labels"]
unique, counts = np.unique(train_labels, return_counts=True)
for u, c in zip(unique, counts):
    print(f"{class_names[u]:>20s} : {c} images")

plt.figure(figsize=(5, 4))
plt.bar([class_names[u] for u in unique], counts, color=["#4C72B0", "#DD8452", "#55A868"])
plt.title("Training set class distribution")
plt.ylabel("Number of images")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
# Visualize a few sample images per class
fig, axes = plt.subplots(num_classes, 4, figsize=(10, 7))
for row, cls_idx in enumerate(range(num_classes)):
    subset = [ex for ex in raw_datasets["train"] if ex["labels"] == cls_idx][:4]
    for col, ex in enumerate(subset):
        axes[row, col].imshow(ex["image"])
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_ylabel(class_names[cls_idx])
    axes[row, 0].set_title(class_names[cls_idx], loc="left")
plt.tight_layout()
plt.show()


## 3. Data Preprocessing & Augmentation

- Images are resized to a fixed `128x128` size (keeps the model small/fast while retaining enough detail for leaf texture).
- **Training** transforms include light augmentation (random flips, rotation, color jitter) to reduce overfitting on this fairly small dataset.
- **Validation/Test** transforms only resize + normalize (no augmentation), so evaluation is on a consistent, non-randomized view of the data.
- Normalization uses standard ImageNet mean/std, a reasonable default for natural RGB images.


In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 32

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

class BeansTorchDataset(Dataset):
    """Wraps a HuggingFace `datasets` split as a PyTorch Dataset."""
    def __init__(self, hf_split, transform):
        self.hf_split = hf_split
        self.transform = transform

    def __len__(self):
        return len(self.hf_split)

    def __getitem__(self, idx):
        example = self.hf_split[idx]
        image = example["image"].convert("RGB")
        label = example["labels"]
        image = self.transform(image)
        return image, label

train_dataset = BeansTorchDataset(raw_datasets["train"], train_transform)
val_dataset = BeansTorchDataset(raw_datasets["validation"], eval_transform)
test_dataset = BeansTorchDataset(raw_datasets["test"], eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

# Sanity check on a batch
imgs, labels = next(iter(train_loader))
print("Batch image tensor shape:", imgs.shape)
print("Batch labels shape:", labels.shape)


## 4. Define CNN Architecture

A custom CNN with **4 convolutional blocks** (Conv → BatchNorm → ReLU → MaxPool), increasing channel depth (32→64→128→256), followed by global average pooling and a small fully-connected classifier head.

**Design rationale:**
- **BatchNorm** after every conv layer stabilizes and speeds up training.
- **Dropout (0.4)** before the final linear layer fights overfitting — important since the dataset is small (~1,000 training images).
- **Global Average Pooling** instead of flattening a large feature map keeps the parameter count (and therefore the saved `.pth` file size) small, and makes the network robust to input size.
- Progressive channel doubling (32→64→128→256) is a standard, well-justified pattern: early layers learn low-level edge/texture features with few channels, deeper layers learn more abstract, class-discriminative features and need more channels to represent them.


In [ ]:
class CustomCNN(nn.Module):
    def __init__(self, num_classes=3, dropout=0.4):
        super().__init__()

        def conv_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),  # halves spatial dimensions
            )

        self.features = nn.Sequential(
            conv_block(3, 32),     # 128 -> 64
            conv_block(32, 64),    # 64  -> 32
            conv_block(64, 128),   # 32  -> 16
            conv_block(128, 256),  # 16  -> 8
        )

        self.global_pool = nn.AdaptiveAvgPool2d(1)  # -> (256, 1, 1)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x


model = CustomCNN(num_classes=num_classes).to(device)
summary(model, input_size=(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE))


In [ ]:
# Rough estimate of saved weight file size, to confirm we stay under the 20MB limit
num_params = sum(p.numel() for p in model.parameters())
approx_size_mb = num_params * 4 / (1024 ** 2)  # float32 = 4 bytes
print(f"Total parameters: {num_params:,}")
print(f"Approx. saved .pth size: {approx_size_mb:.2f} MB")


## 5. Training Loop with Validation

**Hyperparameters and rationale:**
- **Optimizer:** Adam (lr=1e-3) — adapts the learning rate per-parameter, converges faster than plain SGD on small/medium CNNs without heavy tuning.
- **Loss:** CrossEntropyLoss — standard choice for multi-class, single-label classification.
- **LR scheduler:** `ReduceLROnPlateau` on validation loss — automatically lowers the learning rate when validation loss stalls, a simple form of regularization/annealing.
- **Epochs:** 25, with the best model (by validation accuracy) checkpointed — protects against overfitting late in training.
- **Regularization:** Dropout (in the model) + data augmentation (Section 3) + weight decay (1e-4) in the optimizer.


In [ ]:
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 25

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

def run_epoch(loader, model, criterion, optimizer=None):
    """Runs one epoch. If optimizer is provided, trains; otherwise evaluates."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, total_correct, total_samples = 0.0, 0, 0
    torch.set_grad_enabled(is_train)

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        if is_train:
            optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        if is_train:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += images.size(0)

    torch.set_grad_enabled(True)
    return total_loss / total_samples, total_correct / total_samples


In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0
best_model_path = "best_model.pth"

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, model, criterion, optimizer)
    val_loss, val_acc = run_epoch(val_loader, model, criterion, optimizer=None)

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

print(f"\nBest validation accuracy: {best_val_acc:.4f} (checkpoint saved to '{best_model_path}')")


### Training/Validation Loss & Accuracy Curves

In [ ]:
epochs_range = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(epochs_range, history["train_loss"], label="Train Loss")
axes[0].plot(epochs_range, history["val_loss"], label="Val Loss")
axes[0].set_title("Loss vs. Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(epochs_range, history["train_acc"], label="Train Accuracy")
axes[1].plot(epochs_range, history["val_acc"], label="Val Accuracy")
axes[1].set_title("Accuracy vs. Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()


## 6. Evaluate Model on Test Set

We load the **best checkpoint** (highest validation accuracy) and evaluate it once on the held-out test set.

In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

test_acc = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="weighted")

print(f"Test Accuracy : {test_acc:.4f}")
print(f"Precision (weighted): {precision:.4f}")
print(f"Recall (weighted)   : {recall:.4f}")
print(f"F1-score (weighted) : {f1:.4f}")
print()
print("Full classification report:")
print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))


## 7. Visualizations — Confusion Matrix & Per-Class Performance

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.show()


In [ ]:
# Per-class precision/recall/F1
per_class_precision, per_class_recall, per_class_f1, per_class_support = \
    precision_recall_fscore_support(all_labels, all_preds, average=None)

for i, cls in enumerate(class_names):
    print(f"{cls:>20s} | Precision: {per_class_precision[i]:.4f} | "
          f"Recall: {per_class_recall[i]:.4f} | F1: {per_class_f1[i]:.4f} | "
          f"Support: {per_class_support[i]}")

best_idx = int(np.argmax(per_class_f1))
worst_idx = int(np.argmin(per_class_f1))
print(f"\nBest performing class : {class_names[best_idx]} (F1={per_class_f1[best_idx]:.4f})")
print(f"Worst performing class: {class_names[worst_idx]} (F1={per_class_f1[worst_idx]:.4f})")


## 8. Analysis & Discussion of Results

*Fill in the actual numbers you get after running the notebook — the sentences below are a template.*

- **Overall performance:** the model reached a test accuracy of **`<fill in test_acc>`**, with weighted precision/recall/F1 all close to that figure, suggesting the model is not biased heavily toward one class.
- **Best-performing class (`<fill in>`):** likely the class with the most distinctive visual pattern (e.g. `bean_rust`'s reddish-brown pustules are visually very different from a healthy green leaf), which makes it easier for a shallow CNN to separate from the rest.
- **Worst-performing class (`<fill in>`):** often the two disease classes get confused with each other more than with `healthy`, since both present as leaf discoloration/lesions and differ mainly in fine-grained texture — something a deeper/more specialized model or higher input resolution could help resolve.
- **Loss/accuracy curves:** check whether train and validation curves diverge — if train loss keeps dropping while val loss plateaus/rises, that's overfitting; if both stay high, the model may be underfitting (increase depth/capacity or train longer).
- **Effect of augmentation/dropout:** briefly note whether removing them (as an ablation, if you have time) changes the gap between train and validation accuracy.


## 9. Conclusions & Future Work

**Conclusion:** A relatively small, from-scratch CNN (4 conv blocks, global average pooling, dropout) is able to classify bean leaf images into 3 categories (healthy / angular leaf spot / bean rust) with a compact model (~a few MB), well under the 20MB submission limit.

**Possible improvements / future work:**
- **Transfer learning:** fine-tune a pretrained backbone (ResNet18/MobileNetV2) — likely to boost accuracy given the small dataset size.
- **Higher resolution input** (e.g. 224×224) to capture finer lesion texture, at the cost of more compute.
- **More aggressive/targeted augmentation** (e.g. CutMix, MixUp) to further reduce overfitting.
- **Class-specific error analysis:** manually inspect misclassified images between the two disease classes to understand the confusion pattern.
- **K-fold cross-validation** instead of a single train/val split, to get a more robust performance estimate given the modest dataset size.


## Appendix: Saving & Loading the Final Model

The best checkpoint was already saved during training (`best_model.pth`). Below is the standard save/load pattern for `.pth` weight files (see the course's *Model Save and Load Example* for more detail).

In [ ]:
# Final save (state_dict only — smallest file, recommended)
FINAL_MODEL_PATH = "CNN_23-50953-1.pth"
torch.save(model.state_dict(), FINAL_MODEL_PATH)
print(f"Saved model weights to {FINAL_MODEL_PATH} "
      f"({os.path.getsize(FINAL_MODEL_PATH) / (1024**2):.2f} MB)")

# Example of loading it back
loaded_model = CustomCNN(num_classes=num_classes).to(device)
loaded_model.load_state_dict(torch.load(FINAL_MODEL_PATH, map_location=device))
loaded_model.eval()
print("Model successfully reloaded.")
